Заметки:

Обучать модель на **коротких** -> **средних** -> **длинных** последовательностях

Добавлять позиционное кодирование относительно положения токена в слове (Спорные результаты, нужно больше тестов)

Добавлять позиционное кодирование на уровне токенов / на уровне слов

Гипотеза: морфемы слова (приставка, суффикс, окончание) можно выделить в отдельные токены на основе частоты встречаемости соседних букв. Вероятно, что буквы, входящие в приставку, суффикс, окончание, будут соседствовать чаще. Например: "под_пол, под_бежал, зелен_ый, румян_ый, юн_ый..."

Что реализовано:

Неслучайная инициализация параметров модели

Обучаемое и синусоидальное позиционное кодирование (Пока используется синусоидальное для экономии времени обучения и памяти)

Токенизация при помощи BPE. Агрегация субтокенов одного слова простым суммированием, взятием среднего или при помщи внимания. (Внимание показало лучшие результаты)

Простое обучение: вход -> метка.

Результаты на разных конфигурациях:

Синусоидальное позиционное кодирование ВСЕХ токенов, суммирование субтокенов одного слова ~95.29% - 95.32% validation, 60 эпох

Синусоидальное позиционное кодирование ВСЕХ токенов + обучаемое позиционное кодирование всех СУБТОКЕНОВ, суммирование субтокенов одного слова ~94.6% validation, 50 эпох, не увеличивается

Синусоидальное позиционное кодирование ВСЕХ слов (после суммирования субтокенов) + обучаемое позиционное кодирование всех СУБТОКЕНОВ, суммирование субтокенов одного слова ~95.16% validation, 40 эпох

Синусоидальное позиционное кодирование ВСЕХ слов (после суммирования субтокенов), суммирование субтокенов одного слова ~95% validation, 40 эпох

Синусоидальное позиционное кодирование ВСЕХ слов (после суммирования субтокенов), среднее субтокенов одного слова ~93.6% validation, 40 эпох

Обучаемое позиционное кодирование ВСЕХ слов (после суммирования субтокенов), суммирование субтокенов одного слова ~95.33% validation, 60 эпох

Обучаемое позиционное кодирование ВСЕХ токенов, суммирование субтокенов одного слова ~94.65% validation, 40 эпох

Обучаемое позиционное кодирование ВСЕХ токенов, суммирование субтокенов одного слова. Уменьшенный размер словаря (MIN_FRECQUENCY_PAIR = 300, Длина словаря токенов = 1859) ~94% validation, 40 эпох

Обучаемое позиционное кодирование ВСЕХ токенов + обучаемое позиционное кодирование всех СУБТОКЕНОВ, суммирование субтокенов одного слова. Уменьшенный размер словаря (MIN_FRECQUENCY_PAIR = 300, Длина словаря токенов = 1859) ~94.5% validation, 60 эпох

Синусоидальное позиционное кодирование ВСЕХ токенов + обучаемое позиционное кодирование всех СУБТОКЕНОВ, attention суммирование субтокенов одного слова. Cтандартный размер словаря (MIN_FRECQUENCY_PAIR = 100, Длина словаря токенов = 4285) ~96.12% validation, 30 эпох. Быстро переобучается

Синусоидальное позиционное кодирование ВСЕХ токенов + обучаемое позиционное кодирование всех СУБТОКЕНОВ, attention суммирование субтокенов одного слова. Cтандартный размер словаря (MIN_FRECQUENCY_PAIR = 400, Длина словаря токенов = 1452) ~95.95% validation, 30 эпох.

Синусоидальное позиционное кодирование ВСЕХ токенов + обучаемое позиционное кодирование всех СУБТОКЕНОВ + обучаемое позиционное кодирование всех СЛОВ, attention суммирование субтокенов одного слова. Размер словаря (MIN_FRECQUENCY_PAIR = 200, Длина словаря токенов = 2540) ~96.05% validation, 30 эпох.

ИСПОЛЬЗОВАНИЕ АГРЕГАЦИИ ПОСЛЕ MHA

Обучаемое позиционное кодирование ВСЕХ токенов + обучаемое позиционное кодирование всех СУБТОКЕНОВ, суммирование субтокенов одного слова. Cтандартный размер словаря (MIN_FRECQUENCY_PAIR = 100, Длина словаря токенов = 4285) ~95.5% validation, 60 эпох

In [1]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from scripts.custom_dataset import CustomDataset
from model.model import MHAModel
from scripts.tokenizer import train_bpe_tokenizer
from scripts.vectorizer import Vectorizer
from scripts.vocabulary import Vocabulary
import pandas as pd
import numpy as np
import json
import os

In [2]:
UNK_TOKEN = '<UNK>'
MASK_TOKEN = '<MASK>'
PAD_TOKEN = '<PAD>'
BOS_TOKEN='<BOS>'
EOS_TOKEN = '<EOS>'
ADD_BOS_EOS_TOKENS = False

SHUFFLE = True
DROP_LAST = True
EPOCHS = 10
LEARNING_RATE = 1e-5

USE_PRETRAINED = True

BATCH_SIZE = 64
INIT_WEIGHTS = True
BIAS = True
EMBEDDING_DIM = 256
ATTENTION_DIM = 512
NUM_HEADS = 8
NUM_ENCODER_LAYERS = 4
ENCODER_FC_HIDDEN_DIM = ATTENTION_DIM*4 # Как в классическом трансформере
CLASSIFIER_FC_HIDDEN_DIM = ATTENTION_DIM*2

WORDS_POS_ENCODING = 'learnable' # Допустимые значения: sin; learnable; None
TOKENS_POS_ENCODING = 'sin' # Допустимые значения: sin; learnable; None
WORD_SUBTOKENS_POS_ENCODING = 'learnable' # Допустимые значения: learnable; None

SUBTOKENS_AGGREGATION = 'attention' # Допустимые значения: mean; sum; attention
AGGREGATION_MOMENT = 'early' # Допустимые значения: early; late

DROPOUT = 0.15
TEMPERATURE = 1
BATCH_FIRST = True

MODEL_SAVE_FILEPATH = 'data/model_params.pt'
DATASET_PATH = 'D:/Files/Datasets/UD_Russian-SynTagRus-master'

CORPUS_PATH = 'data/corpus.txt'
VOCABULARY_SIZE = 8000
MIN_FRECQUENCY_PAIR = 200

MAX_WORDS_COUNT = -1
MAX_TOKENS_COUNT = 500
MAX_WORD_SUBTOKENS_COUNT = 100

RANDOM_STATE = 42

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# DEVICE = 'cpu'

In [3]:
print(torch.backends.cuda.flash_sdp_enabled())
print(torch.backends.cuda.mem_efficient_sdp_enabled())
print(torch.backends.cuda.math_sdp_enabled())

True
True
True


In [4]:
def find_max_words_source_len(dataframe:pd.DataFrame)->int:
    '''Возвращает максимальную длину НЕ ТОКЕНИЗИРОВАННОЙ входной последовательности в датафрейме'''
    max_words_tokens = 0
    for i in range(len(dataframe)):
        max_words_tokens = max(len(dataframe.loc[i, 'source_words']), max_words_tokens)
    return max_words_tokens

In [5]:
def find_max_tokens_source_len(dataframe:pd.DataFrame, tokenizer)->int:
    '''Возвращает максимальную длину ТОКЕНИЗИРОВАННОЙ входной последовательности в датафрейме'''
    max_source_tokens = 0
    for i in range(len(dataframe)):
        max_source_tokens = max(len(tokenizer.encode(dataframe.loc[i, 'source_text']).tokens), max_source_tokens)
    return max_source_tokens

In [6]:
def generate_batches(dataset:CustomDataset, batch_size:int, shuffle:bool=True, drop_last:bool=True, device='cpu'):
    '''Создает батчи из датасета и переносит данные на девайс.'''
    dataloader = DataLoader(dataset, batch_size, shuffle, drop_last=drop_last)
    for data_dict in dataloader:
        out_data_dict = {}
        for name, _ in data_dict.items():
            out_data_dict[name] = data_dict[name].to(device)
        yield out_data_dict

In [7]:
def save_results_to_file(model, model_filepath:str, train_states:list=None, validation_states:list=None):
    '''Сохраняет параметры модели и метрики обучения в файлы.'''
    torch.save(model, model_filepath)
    if train_states is not None:
        with open("data/train_states.json", "w", encoding="utf-8") as file:
            json.dump(train_states, file, indent=4, ensure_ascii=False)
        
        with open("data/model_hyperparams.json", "w", encoding="utf-8") as file:
            json.dump(model.get_hyperparams(), file, indent=4, ensure_ascii=False)

    if validation_states is not None:
        with open("data/validation_states.json", "w", encoding="utf-8") as file:
            json.dump(validation_states, file, indent=4, ensure_ascii=False)

In [8]:
def preprocess_df(df:pd.DataFrame, source_column_name:str):
    for row in range(len(df)):
        tokens = df[source_column_name].iloc[row].copy()  # Создаем копию
        for i in range(len(tokens)):
            tokens[i] = tokens[i].lower()
        df.at[row, source_column_name] = tokens

In [9]:
def normalize_sizes(predictions:dict[str:torch.tensor], targets:dict[str:torch.tensor], target_names:list[str]):
    for key in target_names:
        # Для predictions: [B, S, C] -> [B*S, C]
        if len(predictions[key].size()) == 3:
            predictions[key] = predictions[key].contiguous().view(-1, predictions[key].size(-1))
        
        # Для targets: [B, S] -> [B*S]
        if len(targets[key].size()) == 2:
            targets[key] = targets[key].contiguous().view(-1)
    
    return predictions, targets

In [10]:
def compute_loss(predictions:dict[str:torch.tensor], targets:dict[str:list[int]], target_names:list[str], target_weights:dict[str:float], pad_idx:int=0):
    predictions, targets = normalize_sizes(predictions, targets, target_names)
    losses = {}
    total_loss = 0
    for key in target_names:
        losses[key] = torch.nn.functional.cross_entropy(predictions[key], targets[key], ignore_index=pad_idx)
        total_loss += losses[key] * target_weights[key]

    return total_loss, losses

In [11]:
def compute_accuracy(predictions:dict[str:torch.tensor], targets:dict[str:list[int]], target_names:list[str], pad_idx:int=0)->dict[str:float]:
    predictions, targets = normalize_sizes(predictions, targets, target_names)
    
    accuracies = {}
    for key in target_names:
        _, pred_indices = predictions[key].max(dim=1)
        
        correct_indices = torch.eq(pred_indices, targets[key]).float()
        valid_indices = torch.ne(targets[key], pad_idx).float()
        
        n_correct = (correct_indices * valid_indices).sum().item()
        n_valid = valid_indices.sum().item()
        accuracies[key] = n_correct / n_valid

    return accuracies

In [12]:
train_df = pd.read_parquet(os.path.join(DATASET_PATH, 'ru_syntagrus-ud-train.parquet'))
validation_df = pd.read_parquet(os.path.join(DATASET_PATH, 'ru_syntagrus-ud-dev.parquet'))

In [13]:
train_df.head(5)

,source_text,source_words,lemmas,upos,xpos,feats,head,deprel,misc
0,Анкета.,"[Анкета, .]","[анкета, .]","[NOUN, PUNCT]","[None, None]","[{'Animacy': 'Inan', 'Case': 'Nom', 'Gender': ...","[0, 1]","[root, punct]","[{'SpaceAfter': 'No'}, None]"
1,Начальник областного управления связи Семен Ер...,"[Начальник, областного, управления, связи, Сем...","[начальник, областной, управление, связь, Семе...","[NOUN, ADJ, NOUN, NOUN, PROPN, PROPN, AUX, NOU...","[None, None, None, None, None, None, None, Non...","[{'Animacy': 'Anim', 'Case': 'Nom', 'Gender': ...","[8, 3, 1, 3, 1, 5, 8, 0, 8, 11, 8, 13, 11, 11,...","[nsubj, amod, nmod, nmod, appos, flat:name, co...","[None, None, None, None, None, None, None, Non..."
2,"В приемной его с утра ожидали посетители, - ко...","[В, приемной, его, с, утра, ожидали, посетител...","[в, приемная, он, с, утро, ожидать, посетитель...","[ADP, NOUN, PRON, ADP, NOUN, VERB, NOUN, PUNCT...","[None, None, None, None, None, None, None, Non...","[None, {'Animacy': 'Inan', 'Case': 'Loc', 'Gen...","[2, 6, 6, 5, 6, 0, 6, 13, 13, 13, 13, 13, 7, 1...","[case, obl, obj, case, obl, root, nsubj, punct...","[None, None, None, None, None, None, {'SpaceAf..."
3,Однако стиль работы Семена Еремеевича заключал...,"[Однако, стиль, работы, Семена, Еремеевича, за...","[однако, стиль, работа, Семен, Еремеевич, закл...","[ADV, NOUN, NOUN, PROPN, PROPN, VERB, ADP, PRO...","[None, None, None, None, None, None, None, Non...","[{'Degree': 'Pos'}, {'Animacy': 'Inan', 'Case'...","[6, 6, 2, 3, 4, 0, 8, 6, 11, 11, 8, 13, 11, 16...","[advmod, nsubj, nmod, nmod, flat:name, root, c...","[None, None, None, None, None, None, None, {'S..."
4,"Приемная была обставлена просто, но по-деловому.","[Приемная, была, обставлена, просто, ,, но, по...","[приемная, быть, обставить, просто, ,, но, по-...","[NOUN, AUX, VERB, ADV, PUNCT, CCONJ, ADV, PUNCT]","[None, None, None, None, None, None, None, None]","[{'Animacy': 'Inan', 'Case': 'Nom', 'Gender': ...","[3, 3, 0, 3, 7, 7, 4, 3]","[nsubj:pass, aux:pass, root, advmod, punct, cc...","[None, None, None, {'SpaceAfter': 'No'}, None,..."


In [14]:
for i in range(len(train_df)):
    row = train_df.iloc[i, :]
    if len(row['source_words']) != len(row['upos']):
        print(f'ne at i = {i}')

In [15]:
tokenizer = train_bpe_tokenizer(['data/corpus.txt'], VOCABULARY_SIZE, MIN_FRECQUENCY_PAIR, unk_token=UNK_TOKEN, pad_token=PAD_TOKEN)
print(len(tokenizer.get_vocab()))

2544


In [16]:
MAX_WORDS_COUNT = max(find_max_words_source_len(train_df), find_max_words_source_len(validation_df))
MAX_TOKENS_COUNT = max(find_max_tokens_source_len(train_df, tokenizer), find_max_tokens_source_len(validation_df, tokenizer))
if ADD_BOS_EOS_TOKENS:
    MAX_TOKENS_COUNT += 2
    MAX_WORDS_COUNT += 2

print(MAX_WORDS_COUNT)
print(MAX_TOKENS_COUNT)

target_names = ['upos']
source_name = 'source_text'
# preprocess_df(train_df, source_name)
# preprocess_df(validation_df, source_name)

205
410


In [17]:
# text = 'Что же ты делаешь? Ты, огузок, отклеивай наклейки от бананов!'
# text = 'Что же ты делаешь ? Ты , огузок , отклеивай наклейки от бананов !'
# tokens = tokenizer.encode(text).tokens
# tokens

In [18]:
source_vocab = Vocabulary(bos_token=BOS_TOKEN, eos_token=EOS_TOKEN, pad_token=PAD_TOKEN, mask_token=MASK_TOKEN, unk_token=UNK_TOKEN, add_bos_eos_tokens=ADD_BOS_EOS_TOKENS)
target_vocabs = {target_name: Vocabulary(bos_token=BOS_TOKEN, eos_token=EOS_TOKEN, pad_token=PAD_TOKEN,\
                                         mask_token=MASK_TOKEN, unk_token=UNK_TOKEN, add_bos_eos_tokens=ADD_BOS_EOS_TOKENS) for target_name in target_names}
for i in range(len(train_df)):
    source_vocab.add_tokens(tokenizer.encode(train_df[source_name].iloc[i]).tokens)
    for target_name in target_names:
        target_vocabs[target_name].add_tokens(train_df[target_name].iloc[i])

pad_idx = source_vocab.pad_idx
source_vocab_len = len(source_vocab)
cls_names_params = {key:len(target_vocabs[key]) for key in target_names}
target_weights = {key : 1.0 for key in target_names}

In [19]:
print(f'Количество батчей = {len(train_df)//BATCH_SIZE}')

print(f'Длина словаря токенов = {len(source_vocab)}')
for key in target_names:
    print(f'Длина словаря признака {key} = {len(target_vocabs[key])}')

Количество батчей = 1087
Длина словаря токенов = 2540
Длина словаря признака upos = 21


In [20]:
if USE_PRETRAINED:
    with open("data/train_states.json", "r", encoding="utf-8") as file:
        train_states = json.load(file)
        training_epochs = int(train_states[-1]['training_epochs'])

    with open("data/validation_states.json", "r", encoding="utf-8") as file:
        validation_states = json.load(file)
    
    model = torch.load(MODEL_SAVE_FILEPATH, weights_only=False)
else:
    train_states = []
    validation_states = []
    training_epochs = 0
    model = MHAModel(MAX_WORDS_COUNT, MAX_TOKENS_COUNT, MAX_WORD_SUBTOKENS_COUNT, source_vocab_len, EMBEDDING_DIM, ATTENTION_DIM, NUM_HEADS, NUM_ENCODER_LAYERS, CLASSIFIER_FC_HIDDEN_DIM, ENCODER_FC_HIDDEN_DIM,\
                     cls_names_params, WORDS_POS_ENCODING, TOKENS_POS_ENCODING, WORD_SUBTOKENS_POS_ENCODING, SUBTOKENS_AGGREGATION, AGGREGATION_MOMENT, DROPOUT, TEMPERATURE, BATCH_FIRST, INIT_WEIGHTS, BIAS, pad_idx)

In [21]:
vectorizer = Vectorizer(tokenizer, source_vocab, target_vocabs, pad_idx)
dataset = CustomDataset(vectorizer, train_df, target_names, MAX_TOKENS_COUNT, MAX_WORDS_COUNT, add_bos_eos_tokens=ADD_BOS_EOS_TOKENS, valid_df=validation_df)

model = model.to(device=DEVICE)
optimizer = optim.Adam(model.parameters(), LEARNING_RATE)

In [22]:
try:
    for epoch in range(EPOCHS):
        training_epochs += 1
        dataset.set_dataframe_split('train')
        batch_generator = generate_batches(dataset, BATCH_SIZE, SHUFFLE, DROP_LAST, DEVICE)
        epoch_sum_train_loss = 0.0
        epoch_running_train_loss = 0.0
        train_running_acc = {key:0.0 for key in target_names}
        model.train()
        for batch_idx, batch_dict in enumerate(batch_generator):

            optimizer.zero_grad()
            
            predictions = model(batch_dict['source_x'], batch_dict['subtokens_cnt'])

            total_loss, train_losses = compute_loss(predictions, batch_dict, target_names, target_weights, pad_idx)

            total_loss.backward()

            optimizer.step()

            # Средние потери и точность
            epoch_running_train_loss += (total_loss.item() - epoch_running_train_loss) / (batch_idx + 1)
            epoch_sum_train_loss += total_loss.item()

            acc_t = compute_accuracy(predictions, batch_dict, target_names, pad_idx)
            for key in target_names:
                train_running_acc[key] += (acc_t[key] - train_running_acc[key]) / (batch_idx + 1)

        dataset.set_dataframe_split('validation')
        batch_generator = generate_batches(dataset, BATCH_SIZE, SHUFFLE, DROP_LAST, DEVICE)
        epoch_sum_valid_loss = 0.0
        epoch_running_valid_loss = 0.0
        valid_running_acc = {key:0.0 for key in target_names}
        model.eval()

        with torch.no_grad():
            for batch_idx, batch_dict in enumerate(batch_generator):
                
                predictions = model(batch_dict['source_x'], batch_dict['subtokens_cnt'])

                total_loss, valid_losses = compute_loss(predictions, batch_dict, target_names, target_weights, pad_idx)

                # Средние потери и точность
                epoch_running_valid_loss += (total_loss.item() - epoch_running_valid_loss) / (batch_idx + 1)
                epoch_sum_valid_loss += total_loss.item()

                acc_t = compute_accuracy(predictions, batch_dict, target_names, pad_idx)
                for key in target_names:
                    valid_running_acc[key] += (acc_t[key] - valid_running_acc[key]) / (batch_idx + 1)

        train_losses_dict = {}
        valid_losses_dict = {}
        for key in train_losses.keys():
            train_losses_dict[key] = train_losses[key].item()
            valid_losses_dict[key] = valid_losses[key].item()

        train_states.append({'training_epochs' : training_epochs, 'epoch_summed_loss' : epoch_sum_train_loss,\
                             'epoch_running_loss' : epoch_running_train_loss, 'losses' : train_losses_dict, 'accuracy' : train_running_acc})
        validation_states.append({'training_epochs' : training_epochs, 'epoch_summed_loss' : epoch_sum_valid_loss,\
                                  'epoch_running_loss' : epoch_running_valid_loss, 'losses' : valid_losses_dict, 'accuracy' : valid_running_acc})
        
        print('-'*40)
        print(f'Epoch {training_epochs}')
        print(f'Train: Суммарная ошибка эпохи {epoch_sum_train_loss}')
        print(f'Train: Средняя ошибка эпохи {epoch_running_train_loss}')
        for key in target_names:
            print(f'Train: Ошибка на признаке {key}: {train_losses_dict[key]}')
            print(f'Train: Точность на признаке {key}: {train_running_acc[key]*100}')

        print('-'*10)
        print(f'Validation: Суммарная ошибка эпохи {epoch_sum_valid_loss}')
        print(f'Validation: Средняя ошибка эпохи {epoch_running_valid_loss}')
        for key in target_names:
            print(f'Validation: Ошибка на признаке {key}: {valid_losses_dict[key]}')
            print(f'Validation: Точность на признаке {key}: {valid_running_acc[key]*100}')

except KeyboardInterrupt:
    print('Принудительная остановка')

----------------------------------------
Epoch 21
Train: Суммарная ошибка эпохи 41.842055991292
Train: Средняя ошибка эпохи 0.038493151785917175
Train: Ошибка на признаке upos: 0.045079175382852554
Train: Точность на признаке upos: 98.52552691021363
----------
Validation: Суммарная ошибка эпохи 20.06210870295763
Validation: Средняя ошибка эпохи 0.14433171728746502
Validation: Ошибка на признаке upos: 0.12219519913196564
Validation: Точность на признаке upos: 96.00678246173538
----------------------------------------
Epoch 22
Train: Суммарная ошибка эпохи 35.15677725337446
Train: Средняя ошибка эпохи 0.03234294135545026
Train: Ошибка на признаке upos: 0.03050176240503788
Train: Точность на признаке upos: 98.76289325683092
----------
Validation: Суммарная ошибка эпохи 20.90285573899746
Validation: Средняя ошибка эпохи 0.15038025711508968
Validation: Ошибка на признаке upos: 0.0869838148355484
Validation: Точность на признаке upos: 96.00672980427497
---------------------------------------

In [23]:
dfdf

NameError: name 'dfdf' is not defined

In [24]:
save_results_to_file(model, MODEL_SAVE_FILEPATH, train_states, validation_states)